# Cerebellar alignment tutorial

Tutorial for cerebellar alignment pipeline.

## This entire notebook should be done in cerebellar_alignment directory for each subject

So make sure all paths specified below are correctly writing images to each subject-week's cerebellar_alignment directory

In [1]:
# if not in same directory as fcn (e.g. avg_vol.py), import cannot find it since notebook is not in root directory of project.
# so add project root
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

In [2]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl

from image_analysis import avg_vol as av
import cerebellum_only_image as coi
import affine_assignment

from pathlib import Path
import os

In [3]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [4]:
#coi.cerebellum_only_img?

### 1. create subfolder (for each subject) for cerebellar-only alignment

subj --> cerebellar_alignment --> [Week_num]

In [5]:
"""
# create cerebellar_alignment subfolder for each subject

for subj in p_df['subj_id'].unique():

    # make directory to store output for each subj
    results_path_dir = Path(anat_dir)/subj/'cerebellar_alignment/'

    # comment out this line after running once, for safety
    #results_path.mkdir(parents=True) # exist_ok = True

    print(f'{subj} cerebellar_alignment directory created!')
"""

"\n# create cerebellar_alignment subfolder for each subject\n\nfor subj in p_df['subj_id'].unique():\n\n    # make directory to store output for each subj\n    results_path_dir = Path(anat_dir)/subj/'cerebellar_alignment/'\n\n    # comment out this line after running once, for safety\n    #results_path.mkdir(parents=True) # exist_ok = True\n\n    print(f'{subj} cerebellar_alignment directory created!')\n"

### 2. cerebellum-only image

In [6]:
# cerebellum-only image
"""
#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # images: t1_anat, cerebel_mask (dseg)

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(mask_path).is_file():
        print(f'mask path does not exist for {subj_id} in week {week}')
        continue
    

    # make a new folder inside subject's week folder for results
    #results_path = Path(anat_dir)/subj_id/week/'cerebellar_alignment/'

    results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week

    # comment out this line after done
    results_path.mkdir(parents=True, exist_ok = True) # only make this directory once!


    # results_path: subj_id, cerebellar_alignment, Wn (create directory ONCE)

    # remove this line after running this function once!!
    #results_path.mkdir(parents=True, exist_ok = True) # only make this directory once!

    #__________________________________

    # function goes here
    coi.cerebellum_only_img(
        cerebellar_mask = mask_path,
        anat_img = t1_path,
        results_path = results_path,
        subj_id = subj_id,
        week = week
    )


    print(f'{subj_id} {week} cerebellar image done')
"""


"\n#_______________________________\n# base loop\nfor i in range(0, p_df.shape[0]):\n    p_id = p_df['ID'].iloc[i]\n    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces\n    p_centre = (str(p_df['Centre'].iloc[i])).strip()\n    refT1 = (p_df['RefT1'].iloc[i]).strip()\n\n    subj_id = f'{p_centre.strip()}_{p_id}'\n\n    # images: t1_anat, cerebel_mask (dseg)\n\n    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'\n    mask_path = f'{anat_dir}/{subj_id}/{week}/iso_norm/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'\n\n    # check that paths exist\n    if not Path(t1_path).is_file():\n        print(f'T1 path does not exist for {subj_id} in week {week}')\n        continue\n\n    if not Path(mask_path).is_file():\n        print(f'mask path does not exist for {subj_id} in week {week}')\n        continue\n    \n\n    # make a new folder inside subject's week folder for results\n    #results_path = Path(anat_dir)/subj_id/week/'cerebellar_alignment/'\n\n   

### 3. SPM coregistration of cerebelli

Use SPM coregistration (function provided in this directory) to coregister each subject week's cerebellum-only image to the reference image.

This will update each (non-reference) cerebellum-only image affine for better alignment of cerebellar voxels.

### 4. Apply new (cerebellar-only-alignment) affine to relevant images

Use the function supplied (`affine_assignment`). This will take the affine from the target image (each week's re-coregistered cerebellum-only image) and assign it to the source image (defined as required).

Save this as a new image in the `cerebellar_alignment` subfolder for each reference week.


In this case (June 12, 2026 at 4:30pm) we want:

- T1_anatomical (native space) (possibly later?)

- white matter probability map (native space)

In [7]:
affine_assignment.affine_assignment?

Signature:
affine_assignment.affine_assignment(
    reference_img,
    source_img,
    results_path,
    update_sform=False,
    update_qform=False,
)
Docstring:
Inputs:
    reference image (Nifti or str): image containing target affine
    source image (Nifti or str): image with affine to update

    results_path (str): path to store image with updated affine

    # taken out for now; testing
    subj_id, week (str)

    update_sform, update_qform: False by default; updates s-form, q-form matrices of source image with that of reference image.

Assigns affine from reference/target image to source image (for world-coordinates alignment).

Output:
    updated affine source image
File:      ~/Documents/GitHub/smarts_cerebellum/cerebellar_alignment/affine_assignment.py
Type:      function

In [8]:
# test this on one subject
p_df = p_df[p_df.subj_id == 'CU_2538']

#### first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
#### and test with the regression function.

In [9]:
# white matter probability images: affine update

#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # images: re-coregistered cerebellum image, white matter probability image.

    t1_source_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    cerebel_ref_path = f'{anat_dir}/{subj_id}/cerebellar_alignment/{week}/{subj_id}_{week}_T1_cerebellum_only.nii'

    # check that paths exist
    if not Path(t1_source_path).is_file():
        print(f'T1 (source) path does not exist for {subj_id} in week {week}')
        continue

    if not Path(cerebel_ref_path).is_file():
        print(f'Cerebellar (ref) path does not exist for {subj_id} in week {week}')
        continue

    results_path = Path(anat_dir)/subj_id/'cerebellar_alignment'/week
    
    #__________________________________
    
    # function goes here

    # first try (on test subject) without updating sform, qform; then check these matrices (on updated image in cerebellar_alignment dir) against original (in subj_id/week dir) (not updated affines)
    # and test with the regression function.
    affine_assignment.affine_assignment(
        reference_img = cerebel_ref_path,
        source_img = t1_source_path,

        results_path = results_path,
    )
    

    print(f'{subj_id} {week} wm_native affine updated!')


TypeError: expected str, bytes or os.PathLike object, not Nifti1Image

the issue there if with trying to get the path to the source_name, so we can either:
(a) figure this out (get name of nifti img) or
(b) just manually put in the name in the function

I think I will make the output just the image, and then save the image in the tutorial (in this loop)

In [ ]:
t1_source_path

'/cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W0/CU_2538_W0_T1.nii'

### 5. Apply linear regression (voxel-wise)

Use the supplied function, `avg_vol`, to apply linear regression in native space.

For now, we will just do this on the white matter probability images.

# check results path

run on test subj

In [ ]:
# CALL FOR WM SEGMENTATION IMAGE IN NATIVE SPACE________

betas = [] # store matrices for all subjects

for subj in p_df['subj_id'].unique():

    ref_img = f'{anat_dir}/{subj}/{refT1}/c2{subj}_{refT1}_T1.nii'

    results_path = f'{anat_dir}/{subj}/cerebellar_alignment'

    betas.append(av.avg_vol(subj_id = subj, reference_img=ref_img,
                         results_path = results_path
                         ))
    
    # get a single slope and intercept image for each subject

# check results path again after run on test subj

### 6. reslice to MNISymm space

Reslice slope images for each subject to group space (from native - cerebellar-aligned - space to a group template, MNISymm)

### 7. Mirror lesion

for those with lesions in the left cerebral hemisphere, mirror their cerebellum slope image (in template space - symmetric) to the right hemisphere.
Since this is a symmetric template, we can just flip it along the x-axis (L-R flip).

### 8. Get average (or sum?) slope image for each of patients and controls

See where we have change in cerebellum across all subjects - either take the average or the sum (or even median?)

Function to do this coming soon! (where function's arguments should include 'stat')